In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Dense,Dropout



In [ ]:
data= pd.read_csv('Google_Stock_Price_Train.csv', thousands=",")
data

In [ ]:
data.info()

In [ ]:
data['Date']=pd.to_datetime(data['Date'])

In [ ]:
fig = data.plot(
    x="Date",
    y=["Open","High","Low","Close"],
    figsize=(12,6),
    title="Open, High, Low, Close Prices of Google's stocks"
)

fig.set_xlabel("Date")
fig.set_ylabel("Stock Proce")
plt.grid(True)
plt.show()


In [ ]:
data = data.drop(['Date'],axis=1)

In [ ]:
sc= MinMaxScaler()
data_sc = sc.fit_transform(data) 

In [ ]:
def create_sequences(data,target_col_idx,seq_len=60):
    x,y=[],[]
    for i in range(seq_len,len(data)):
        x.append(data[i-seq_len:i])
        y.append(data[i,target_col_idx])
    return np.array(x),np.array(y)
x,y = create_sequences(data_sc,3)

In [ ]:
x

In [ ]:
split = int(len(x)*0.8)
x_train,x_test = x[:split], x[split:]
y_train,y_test = y[:split], y[split:]

In [ ]:
model=Sequential()
model.add(LSTM(64,return_sequences=True,input_shape=(x_train.shape[1],x_train.shape[2])))
model.add(Dropout(0.2))
model.add(LSTM(64))
model.add(Dropout(0.1))
model.add(Dense(1))
model.compile(optimizer='adam',loss='mean_squared_error')
model.summary()

In [ ]:
h = model.fit(
    x_train,y_train,
    epochs=50,
    batch_size=32,
    validation_data=(x_test,y_test)
)

In [ ]:
y_pred = model.predict(x_test)


In [ ]:
close_scaler =MinMaxScaler()
close_scaler.min_,close_scaler.scale_ = sc.min_[3:4],sc.scale_[3:4]

In [ ]:
predicted_values = close_scaler.inverse_transform(y_pred)
actual_values = close_scaler.inverse_transform(y_test.reshape(-1,1))

In [ ]:

plt.figure(figsize=(12, 6))
plt.plot(actual_values, label='actual')
plt.plot(predicted_values, label='predicted')
plt.title("Google Stock Price Prediction")
plt.xlabel("Time")
plt.ylabel("Close Price")
plt.legend()
plt.show()

In [ ]:
rmse = np.sqrt(mean_squared_error(actual_values, predicted_values))
mae = mean_absolute_error(actual_values, predicted_values)
r2 = r2_score(actual_values, predicted_values)

print('RMSE: ', rmse)
print('MAE: ', mae)
print('R2 Score: ', r2)